In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)

ROOT = Path('/content/drive/MyDrive/TrafficAI_ControlCenter')
DATA = ROOT / 'datasets' / 'processed' / 'module2_breakdown' / 'module2_ml_table.parquet'
MODEL_DIR = ROOT / 'models' / 'module2' / 'baselines'
MET_DIR = ROOT / 'metrics' / 'module2'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MET_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(DATA)
print(df['y'].value_counts())
feature_cols = json.loads((ROOT/'datasets'/'processed'/'module2_breakdown'/'module2_ml_meta.json').read_text())['features']
X = df[feature_cols].fillna(df[feature_cols].median(numeric_only=True))
y = df['y'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

Mounted at /content/drive
y
0    12838
1     1272
Name: count, dtype: int64
(11288, 12) (2822, 12)


In [2]:
!pip -q install xgboost joblib

In [3]:
def eval_model(name, model, X_te, y_te):
    pred = model.predict(X_te)
    metrics = {
        'model': name,
        'category': 'Base ML',
        'module': 'module2_breakdown',
        'accuracy': float(accuracy_score(y_te, pred)),
        'precision': float(precision_score(y_te, pred, average='binary', zero_division=0)),
        'recall': float(recall_score(y_te, pred, average='binary', zero_division=0)),
        'f1': float(f1_score(y_te, pred, average='binary', zero_division=0)),
    }
    try:
        proba = model.predict_proba(X_te)[:,1]
        metrics['roc_auc'] = float(roc_auc_score(y_te, proba))
    except Exception:
        metrics['roc_auc'] = None
    print('\n', name)
    print(classification_report(y_te, pred, digits=4, zero_division=0))
    print(metrics)
    return metrics

# 0 LR
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=2000, class_weight='balanced'))
])
pipe_lr.fit(X_train, y_train)
m0 = eval_model('Logistic_Regression', pipe_lr, X_test, y_test)
joblib.dump(pipe_lr, MODEL_DIR/'logistic_regression.joblib')
(MET_DIR/'Logistic_Regression.json').write_text(json.dumps(m0, indent=2))

# 1 RF
rf = RandomForestClassifier(
    n_estimators=300, max_depth=12, min_samples_leaf=3,
    class_weight='balanced_subsample', random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
m1 = eval_model('Random_Forest', rf, X_test, y_test)
joblib.dump(rf, MODEL_DIR/'random_forest.joblib')
(MET_DIR/'Random_Forest.json').write_text(json.dumps(m1, indent=2))

# 2 GB
vc = y_train.value_counts()
w_map = {c: len(y_train)/(len(vc)*cnt) for c,cnt in vc.items()}
sw = y_train.map(w_map)
gb = GradientBoostingClassifier(random_state=42, max_depth=3, learning_rate=0.08, n_estimators=200)
gb.fit(X_train, y_train, sample_weight=sw)
m2 = eval_model('Gradient_Boosting', gb, X_test, y_test)
joblib.dump(gb, MODEL_DIR/'gradient_boosting.joblib')
(MET_DIR/'Gradient_Boosting.json').write_text(json.dumps(m2, indent=2))

# 3 XGB
from xgboost import XGBClassifier
spw = (y_train==0).sum() / max((y_train==1).sum(), 1)
xgb = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.08,
    subsample=0.9, colsample_bytree=0.9,
    objective='binary:logistic', eval_metric='auc',
    scale_pos_weight=spw, tree_method='hist',
    random_state=42, n_jobs=-1
)
xgb.fit(X_train, y_train)
m3 = eval_model('XGBoost', xgb, X_test, y_test)
joblib.dump(xgb, MODEL_DIR/'xgboost.joblib')
(MET_DIR/'XGBoost.json').write_text(json.dumps(m3, indent=2))

summary = [m0,m1,m2,m3]
pd.DataFrame(summary).to_csv(MET_DIR/'baselines_summary.csv', index=False)
(MET_DIR/'baselines_summary.json').write_text(json.dumps(summary, indent=2))
print(pd.DataFrame(summary)[['model','accuracy','precision','recall','f1','roc_auc']])
print('STEP 7a OK')


 Logistic_Regression
              precision    recall  f1-score   support

           0     1.0000    0.9412    0.9697      2568
           1     0.6272    1.0000    0.7709       254

    accuracy                         0.9465      2822
   macro avg     0.8136    0.9706    0.8703      2822
weighted avg     0.9664    0.9465    0.9518      2822

{'model': 'Logistic_Regression', 'category': 'Base ML', 'module': 'module2_breakdown', 'accuracy': 0.946491849751949, 'precision': 0.6271604938271605, 'recall': 1.0, 'f1': 0.7708649468892261, 'roc_auc': 0.9861008904260798}

 Random_Forest
              precision    recall  f1-score   support

           0     0.9976    0.9829    0.9902      2568
           1     0.8493    0.9764    0.9084       254

    accuracy                         0.9823      2822
   macro avg     0.9235    0.9796    0.9493      2822
weighted avg     0.9843    0.9823    0.9828      2822

{'model': 'Random_Forest', 'category': 'Base ML', 'module': 'module2_breakdown', 'acc

In [3]:
import torch, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

assert torch.cuda.is_available(), 'Enable T4 GPU'
device = torch.device('cuda')
print(torch.cuda.get_device_name(0))

ROOT = Path('/content/drive/MyDrive/TrafficAI_ControlCenter')
PROC = ROOT / 'datasets' / 'processed' / 'module2_breakdown'
MODEL_DIR = ROOT / 'models' / 'module2' / 'dl'
MET_DIR = ROOT / 'metrics' / 'module2'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(PROC / 'module2_ml_table.parquet').sort_values(['video_id','t_sec'])
feat_cols = json.loads((PROC/'module2_ml_meta.json').read_text())['features']
df[feat_cols] = df[feat_cols].fillna(df[feat_cols].median(numeric_only=True))

SEQ_LEN = 12
Xs, ys = [], []
for vid, g in df.groupby('video_id', sort=False):
    if len(g) < SEQ_LEN:
        continue
    arr = g[feat_cols].to_numpy(np.float32)
    lab = g['y'].to_numpy(np.int64)
    for i in range(SEQ_LEN-1, len(g)):
        Xs.append(arr[i-SEQ_LEN+1:i+1])
        ys.append(lab[i])

X = np.stack(Xs); y = np.array(ys)
print('seq', X.shape, 'pos', y.sum(), 'neg', (y==0).sum())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
mean = X_train.mean((0,1), keepdims=True); std = X_train.std((0,1), keepdims=True)+1e-6
X_train = (X_train-mean)/std; X_test = (X_test-mean)/std
np.save(MODEL_DIR/'seq_mean.npy', mean); np.save(MODEL_DIR/'seq_std.npy', std)

class SeqDS(Dataset):
    def __init__(self, X, y):
        self.X, self.y = torch.tensor(X), torch.tensor(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_loader = DataLoader(SeqDS(X_train,y_train), batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(SeqDS(X_test,y_test), batch_size=512, shuffle=False, num_workers=2, pin_memory=True)

# binary class weight
neg, pos = (y_train==0).sum(), max((y_train==1).sum(),1)
pos_weight = torch.tensor([neg/pos], device=device, dtype=torch.float32)
print('pos_weight', float(pos_weight))

Tesla T4


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/TrafficAI_ControlCenter/datasets/processed/module2_breakdown/module2_ml_table.parquet'

In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
ROOT = Path('/content/drive/MyDrive/TrafficAI_ControlCenter')

hits = list(ROOT.rglob('*module2*'))
print('module2-related paths:')
for p in hits:
    print(' -', p)

ValueError: Mountpoint must not already contain files

In [5]:
from google.colab import drive
from pathlib import Path
import os

# mount only if needed
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print('Drive already mounted — skipping remount')

ROOT = Path('/content/drive/MyDrive/TrafficAI_ControlCenter')
print('ROOT exists:', ROOT.exists())

print('\nmodule2-related:')
for p in ROOT.rglob('*module2*'):
    print(' -', p)

print('\nml tables:')
for p in ROOT.rglob('module2_ml_table*'):
    print(' FOUND', p, 'size', p.stat().st_size)

Drive already mounted — skipping remount
ROOT exists: True

module2-related:
 - /content/drive/MyDrive/TrafficAI_ControlCenter/models/module2

ml tables:


In [6]:
import json
import pandas as pd

PROC = ROOT / 'datasets' / 'processed' / 'module2_breakdown'
parquet = PROC / 'module2_ml_table.parquet'
csv = PROC / 'module2_ml_table_sample.csv'

if parquet.exists():
    df = pd.read_parquet(parquet)
    print('Loaded parquet', df.shape)
elif csv.exists():
    df = pd.read_csv(csv)
    print('Loaded csv', df.shape)
else:
    alts = list(ROOT.rglob('module2_ml_table.parquet')) + list(ROOT.rglob('module2_ml_table*.csv'))
    print('alts', alts)
    if not alts:
        raise FileNotFoundError('Module2 table missing — re-run Step 6 save')
    p = alts[0]
    df = pd.read_parquet(p) if p.suffix == '.parquet' else pd.read_csv(p)
    print('Loaded alt', p, df.shape)

print(df['y'].value_counts())
meta = PROC / 'module2_ml_meta.json'
feat_cols = json.loads(meta.read_text())['features'] if meta.exists() else [c for c in df.columns if c not in ['y','video_id','video_path']]
print('features', feat_cols)

alts []


FileNotFoundError: Module2 table missing — re-run Step 6 save

In [7]:
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/TrafficAI_ControlCenter')

print('processed tree:')
proc = ROOT / 'datasets' / 'processed'
if proc.exists():
    for p in sorted(proc.rglob('*')):
        if p.is_file():
            print(p.relative_to(ROOT), p.stat().st_size)
else:
    print('processed/ missing')

print('\nanomaly files:')
for p in ROOT.rglob('*anomaly*'):
    print(' -', p)

processed tree:
processed/ missing

anomaly files:


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/TrafficAI_ControlCenter')
print('ROOT', ROOT.exists(), ROOT)

proc = ROOT / 'datasets' / 'processed'
print('processed exists:', proc.exists())

if proc.exists():
    for p in sorted(proc.rglob('*')):
        if p.is_file():
            print(p.relative_to(ROOT), p.stat().st_size)

ROOT True /content/drive/MyDrive/TrafficAI_ControlCenter
processed exists: True
datasets/processed/module1_bus/bus_gps_points.parquet 3295351
datasets/processed/module1_bus/bus_gps_points_sample.csv 881738
datasets/processed/module1_bus/gtfs_node_map_route_425.csv 1386
datasets/processed/module1_bus/gtfs_node_map_route_534.csv 1186
datasets/processed/module1_bus/gtfs_routes.parquet 40129
datasets/processed/module1_bus/gtfs_stop_times.parquet 18500250
datasets/processed/module1_bus/gtfs_stops.parquet 256494
datasets/processed/module1_bus/gtfs_trips.parquet 553557
datasets/processed/module1_bus/module1_ml_meta.json 330
datasets/processed/module1_bus/module1_ml_table.parquet 3457199
datasets/processed/module1_bus/module1_ml_table_sample.csv 38665026
datasets/processed/module2_breakdown/anomaly_events.csv 2881
datasets/processed/module2_breakdown/anomaly_events.parquet 5194
datasets/processed/module2_breakdown/module2_ml_meta.json 540
datasets/processed/module2_breakdown/module2_ml_table.p

In [3]:
import pandas as pd, json
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/TrafficAI_ControlCenter')
PROC = ROOT / 'datasets' / 'processed' / 'module2_breakdown'

df = pd.read_parquet(PROC / 'module2_ml_table.parquet')
feat_cols = json.loads((PROC / 'module2_ml_meta.json').read_text())['features']
print(df.shape)
print(df['y'].value_counts())
print('features', feat_cols)

(14110, 14)
y
0    12838
1     1272
Name: count, dtype: int64
features ['t_sec', 'duration_sec', 'rel_time', 'fps', 'motion_mean', 'motion_std', 'motion_p95', 'stationary_score', 'motion_mean_z', 'low_motion_flag', 'rule_stop_hint', 'has_any_anomaly_in_video']


In [4]:
import torch, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

assert torch.cuda.is_available(), 'Enable T4 GPU'
device = torch.device('cuda')
print(torch.cuda.get_device_name(0))

ROOT = Path('/content/drive/MyDrive/TrafficAI_ControlCenter')
PROC = ROOT / 'datasets' / 'processed' / 'module2_breakdown'
MODEL_DIR = ROOT / 'models' / 'module2' / 'dl'
MET_DIR = ROOT / 'metrics' / 'module2'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(PROC / 'module2_ml_table.parquet').sort_values(['video_id','t_sec'])
feat_cols = json.loads((PROC/'module2_ml_meta.json').read_text())['features']
df[feat_cols] = df[feat_cols].fillna(df[feat_cols].median(numeric_only=True))

SEQ_LEN = 12
Xs, ys = [], []
for vid, g in df.groupby('video_id', sort=False):
    if len(g) < SEQ_LEN:
        continue
    arr = g[feat_cols].to_numpy(np.float32)
    lab = g['y'].to_numpy(np.int64)
    for i in range(SEQ_LEN-1, len(g)):
        Xs.append(arr[i-SEQ_LEN+1:i+1])
        ys.append(lab[i])

X = np.stack(Xs); y = np.array(ys)
print('seq', X.shape, 'pos', y.sum(), 'neg', (y==0).sum())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
mean = X_train.mean((0,1), keepdims=True); std = X_train.std((0,1), keepdims=True)+1e-6
X_train = (X_train-mean)/std; X_test = (X_test-mean)/std
np.save(MODEL_DIR/'seq_mean.npy', mean); np.save(MODEL_DIR/'seq_std.npy', std)

class SeqDS(Dataset):
    def __init__(self, X, y):
        self.X, self.y = torch.tensor(X), torch.tensor(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_loader = DataLoader(SeqDS(X_train,y_train), batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(SeqDS(X_test,y_test), batch_size=512, shuffle=False, num_workers=2, pin_memory=True)

# binary class weight
neg, pos = (y_train==0).sum(), max((y_train==1).sum(),1)
pos_weight = torch.tensor([neg/pos], device=device, dtype=torch.float32)
print('pos_weight', float(pos_weight))

Tesla T4
seq (13230, 12, 12) pos 1239 neg 11991
pos_weight 9.680121421813965


In [5]:
F = len(feat_cols)

class AttnPool(nn.Module):
    def __init__(self, d):
        super().__init__(); self.w = nn.Linear(d,1)
    def forward(self, h):
        a = torch.softmax(self.w(h), dim=1)
        return (a*h).sum(1)

def evaluate(model):
    model.eval(); preds=[]; trues=[]; probs=[]
    with torch.no_grad():
        for xb,yb in test_loader:
            xb = xb.to(device)
            logit = model(xb).squeeze(1)
            prob = torch.sigmoid(logit).cpu().numpy()
            pred = (prob>=0.5).astype(int)
            preds.append(pred); trues.append(yb.numpy()); probs.append(prob)
    yt, yp, pr = np.concatenate(trues), np.concatenate(preds), np.concatenate(probs)
    return {
        'accuracy': float(accuracy_score(yt,yp)),
        'precision': float(precision_score(yt,yp, zero_division=0)),
        'recall': float(recall_score(yt,yp, zero_division=0)),
        'f1': float(f1_score(yt,yp, zero_division=0)),
        'roc_auc': float(roc_auc_score(yt,pr)),
    }

def train_model(model, name, epochs=8, lr=1e-3):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    best_f1, best= -1, None
    for ep in range(1, epochs+1):
        model.train(); total=0.0
        for xb,yb in train_loader:
            xb,yb = xb.to(device), yb.float().to(device)
            opt.zero_grad()
            loss = crit(model(xb).squeeze(1), yb)
            loss.backward(); opt.step()
            total += loss.item()*len(yb)
        met = evaluate(model)
        print(f'{name} epoch {ep}/{epochs} loss={total/len(y_train):.4f} f1={met["f1"]:.4f} auc={met["roc_auc"]:.4f}')
        if met['f1'] > best_f1:
            best_f1 = met['f1']
            best = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(best)
    met = evaluate(model)
    met.update({'model':name,'category':'Optimized DL','module':'module2_breakdown'})
    torch.save({'state_dict':best,'feat_cols':feat_cols,'seq_len':SEQ_LEN}, MODEL_DIR/f'{name}.pt')
    (MET_DIR/f'{name}.json').write_text(json.dumps(met, indent=2))
    print('Saved', name, met)
    return met

In [6]:
class LSTMAttn(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm=nn.LSTM(F,64,batch_first=True)
        self.attn=AttnPool(64); self.fc=nn.Linear(64,1)
    def forward(self,x):
        h,_=self.lstm(x); return self.fc(self.attn(h))

class BiLSTMAttn(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm=nn.LSTM(F,64,batch_first=True,bidirectional=True)
        self.attn=AttnPool(128); self.fc=nn.Linear(128,1)
    def forward(self,x):
        h,_=self.lstm(x); return self.fc(self.attn(h))

class TCNAttn(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1=nn.Conv1d(F,64,3,padding=1)
        self.c2=nn.Conv1d(64,64,3,padding=1)
        self.c3=nn.Conv1d(64,64,3,padding=1)
        self.attn=AttnPool(64); self.fc=nn.Linear(64,1)
    def forward(self,x):
        z=x.transpose(1,2)
        z=torch.relu(self.c3(torch.relu(self.c2(torch.relu(self.c1(z)))))).transpose(1,2)
        return self.fc(self.attn(z))

class FeatureAttn(nn.Module):
    def __init__(self):
        super().__init__()
        self.feat=nn.Sequential(nn.Linear(F,64),nn.ReLU(),nn.Linear(64,F),nn.Sigmoid())
        self.proj=nn.Linear(F,64)
        layer=nn.TransformerEncoderLayer(64,4,128,batch_first=True)
        self.enc=nn.TransformerEncoder(layer,2)
        self.attn=AttnPool(64); self.fc=nn.Linear(64,1)
    def forward(self,x):
        x=x*self.feat(x[:,-1,:]).unsqueeze(1)
        h=self.enc(self.proj(x)); return self.fc(self.attn(h))

class TransformerCW(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj=nn.Linear(F,64)
        layer=nn.TransformerEncoderLayer(64,4,128,batch_first=True)
        self.enc=nn.TransformerEncoder(layer,3)
        self.fc=nn.Linear(64,1)
    def forward(self,x):
        return self.fc(self.enc(self.proj(x))[:,-1,:])

class GATGRU(nn.Module):
    def __init__(self):
        super().__init__()
        self.in_proj=nn.Linear(F,32)
        self.attn_l=nn.Linear(64,1)
        self.gru=nn.GRU(32,64,batch_first=True)
        self.fc=nn.Linear(64,1)
    def forward(self,x):
        b,t,_=x.shape
        h=torch.relu(self.in_proj(x))
        outs=[]
        idx=torch.arange(t, device=x.device).float()
        for i in range(t):
            center=h[:,i:i+1,:].expand(-1,t,-1)
            score=self.attn_l(torch.cat([center,h],-1)).squeeze(-1)
            score=score-((idx-i).abs().unsqueeze(0)/max(t-1,1))
            a=torch.softmax(score,1).unsqueeze(-1)
            outs.append((a*h).sum(1))
        g=torch.stack(outs,1)
        y,_=self.gru(g)
        return self.fc(y[:,-1,:])

class PolicyHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone=BiLSTMAttn()
        self.backbone.fc=nn.Identity()
        self.policy=nn.Sequential(nn.Linear(128,64),nn.ReLU(),nn.Dropout(0.2),nn.Linear(64,1))
    def forward(self,x):
        return self.policy(self.backbone(x))

In [7]:
class LSTMAttn(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm=nn.LSTM(F,64,batch_first=True)
        self.attn=AttnPool(64); self.fc=nn.Linear(64,1)
    def forward(self,x):
        h,_=self.lstm(x); return self.fc(self.attn(h))

class BiLSTMAttn(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm=nn.LSTM(F,64,batch_first=True,bidirectional=True)
        self.attn=AttnPool(128); self.fc=nn.Linear(128,1)
    def forward(self,x):
        h,_=self.lstm(x); return self.fc(self.attn(h))

class TCNAttn(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1=nn.Conv1d(F,64,3,padding=1)
        self.c2=nn.Conv1d(64,64,3,padding=1)
        self.c3=nn.Conv1d(64,64,3,padding=1)
        self.attn=AttnPool(64); self.fc=nn.Linear(64,1)
    def forward(self,x):
        z=x.transpose(1,2)
        z=torch.relu(self.c3(torch.relu(self.c2(torch.relu(self.c1(z)))))).transpose(1,2)
        return self.fc(self.attn(z))

class FeatureAttn(nn.Module):
    def __init__(self):
        super().__init__()
        self.feat=nn.Sequential(nn.Linear(F,64),nn.ReLU(),nn.Linear(64,F),nn.Sigmoid())
        self.proj=nn.Linear(F,64)
        layer=nn.TransformerEncoderLayer(64,4,128,batch_first=True)
        self.enc=nn.TransformerEncoder(layer,2)
        self.attn=AttnPool(64); self.fc=nn.Linear(64,1)
    def forward(self,x):
        x=x*self.feat(x[:,-1,:]).unsqueeze(1)
        h=self.enc(self.proj(x)); return self.fc(self.attn(h))

class TransformerCW(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj=nn.Linear(F,64)
        layer=nn.TransformerEncoderLayer(64,4,128,batch_first=True)
        self.enc=nn.TransformerEncoder(layer,3)
        self.fc=nn.Linear(64,1)
    def forward(self,x):
        return self.fc(self.enc(self.proj(x))[:,-1,:])

class GATGRU(nn.Module):
    def __init__(self):
        super().__init__()
        self.in_proj=nn.Linear(F,32)
        self.attn_l=nn.Linear(64,1)
        self.gru=nn.GRU(32,64,batch_first=True)
        self.fc=nn.Linear(64,1)
    def forward(self,x):
        b,t,_=x.shape
        h=torch.relu(self.in_proj(x))
        outs=[]
        idx=torch.arange(t, device=x.device).float()
        for i in range(t):
            center=h[:,i:i+1,:].expand(-1,t,-1)
            score=self.attn_l(torch.cat([center,h],-1)).squeeze(-1)
            score=score-((idx-i).abs().unsqueeze(0)/max(t-1,1))
            a=torch.softmax(score,1).unsqueeze(-1)
            outs.append((a*h).sum(1))
        g=torch.stack(outs,1)
        y,_=self.gru(g)
        return self.fc(y[:,-1,:])

class PolicyHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone=BiLSTMAttn()
        self.backbone.fc=nn.Identity()
        self.policy=nn.Sequential(nn.Linear(128,64),nn.ReLU(),nn.Dropout(0.2),nn.Linear(64,1))
    def forward(self,x):
        return self.policy(self.backbone(x))

In [8]:
results=[]
results.append(train_model(LSTMAttn(), 'LSTM_Attention', epochs=8))
results.append(train_model(BiLSTMAttn(), 'BiLSTM_Attention', epochs=8))
results.append(train_model(TCNAttn(), 'TCN_Attention', epochs=8))
results.append(train_model(FeatureAttn(), 'Transformer_Feature_Attention', epochs=8))
results.append(train_model(TransformerCW(), 'Transformer_Encoder_ClassWeighted', epochs=8))
results.append(train_model(GATGRU(), 'GAT_GRU', epochs=6))
results.append(train_model(PolicyHead(), 'Decision_Policy_Head', epochs=8))

sum_df=pd.DataFrame(results)[['model','category','accuracy','precision','recall','f1','roc_auc']]
print(sum_df.sort_values('f1', ascending=False))
sum_df.to_csv(MET_DIR/'dl_summary.csv', index=False)
(MET_DIR/'dl_summary.json').write_text(json.dumps(results, indent=2))

base=json.loads((MET_DIR/'baselines_summary.json').read_text())
arena=base+results
pd.DataFrame(arena).to_csv(MET_DIR/'arena_module2_11models.csv', index=False)
(MET_DIR/'arena_module2_11models.json').write_text(json.dumps(arena, indent=2))
print('STEP 7b OK — Module 2 eleven-model arena saved')

LSTM_Attention epoch 1/8 loss=0.9058 f1=0.7619 auc=0.9750
LSTM_Attention epoch 2/8 loss=0.3089 f1=0.7619 auc=0.9781
LSTM_Attention epoch 3/8 loss=0.2155 f1=0.7619 auc=0.9788
LSTM_Attention epoch 4/8 loss=0.2082 f1=0.7619 auc=0.9794
LSTM_Attention epoch 5/8 loss=0.2043 f1=0.7774 auc=0.9800
LSTM_Attention epoch 6/8 loss=0.2013 f1=0.7714 auc=0.9806
LSTM_Attention epoch 7/8 loss=0.2009 f1=0.7666 auc=0.9812
LSTM_Attention epoch 8/8 loss=0.1972 f1=0.7786 auc=0.9818
Saved LSTM_Attention {'accuracy': 0.9467120181405896, 'precision': 0.6375321336760925, 'recall': 1.0, 'f1': 0.7786499215070644, 'roc_auc': 0.9817943043934462, 'model': 'LSTM_Attention', 'category': 'Optimized DL', 'module': 'module2_breakdown'}
BiLSTM_Attention epoch 1/8 loss=0.8546 f1=0.7492 auc=0.9740
BiLSTM_Attention epoch 2/8 loss=0.2340 f1=0.7619 auc=0.9806
BiLSTM_Attention epoch 3/8 loss=0.2057 f1=0.7750 auc=0.9809
BiLSTM_Attention epoch 4/8 loss=0.1983 f1=0.7774 auc=0.9830
BiLSTM_Attention epoch 5/8 loss=0.1918 f1=0.7848 au